# 07 · Preparação para Machine Learning

Notebook **utilitário**, chamado via `%run` pelos notebooks de treino
(`08` a `11`) — não é uma etapa do pipeline em si, é onde ficam as
funções compartilhadas entre todos os modelos, pra não duplicar essa
lógica em cada notebook de treino:

1. **`carregar_features_com_calendario(nome_tabela)`** — lê uma tabela
`silver.features_*` e junta os atributos de calendário
(`silver.features_calendario`: dia da semana, trimestre, feriado...),
que ficaram propositalmente separados até aqui (ver notebook `05`).
2. **`split_temporal(df, ...)`** — divide qualquer tabela de série em
treino/validação/teste **sequencialmente** (não aleatório — embaralhar
vazaria "futuro" pros lags/médias móveis, que são calculados sobre a
linha do tempo inteira), com um **embargo** entre os conjuntos.

**Por que o embargo:** o alvo `target_d7` de uma linha "enxerga" 7 dias à
frente dela. Uma linha de treino perto da fronteira do conjunto de teste
teria seu target calculado usando dados que já estão dentro da janela de
teste — vazamento de informação do futuro para o passado. O embargo
remove um pequeno intervalo entre os conjuntos pra eliminar essa
sobreposição, prática padrão em séries temporais.

In [ ]:
%run ./00_config

In [ ]:
from pyspark.sql import functions as F, DataFrame
from datetime import timedelta

## `carregar_features_com_calendario`

Junta qualquer tabela `silver.features_*` (série de volume ou de risco
de OLA) com `silver.features_calendario` pela chave `data_abertura`. As
duas cobrem exatamente o mesmo range de datas (mesma lógica de
`DATA_MIN`/`DATA_MAX` calculada no notebook `05`), então o join não deve
gerar nenhum nulo novo — validamos isso explicitamente.

**Corte de regime não-estacionário:** 2023 até nov/2024 representa
66,7% do período mas só **1,73%** do volume de incidentes — ruído de
adoção da ferramenta de ITSM, não sinal operacional real (achado do
notebook `04`). Deixar isso no treino ensinaria o modelo a enxergar uma
"tendência de crescimento explosivo" que não existe mais — é ruído
enganoso, não histórico neutro. Por isso, por padrão, esta função já
corta a série a partir de **dezembro/2024** antes de devolver o
DataFrame — vale para **todo** modelo treinado a partir daqui (volume e
risco de OLA), sem precisar repetir o filtro em cada notebook.

O corte é aplicado **depois** do join com o calendário — as flags
`is_feriado`/`is_fim_de_semana` continuam presentes em toda linha que
sobra, não se perdem no processo.

As tabelas Silver em si **não são alteradas** — continuam com o
histórico completo (2023-2025) para contexto/auditoria/BI. O corte é
só no que efetivamente vira dado de treino.

In [ ]:
DATA_INICIO_TREINO_ML = "2024-12-01"

In [ ]:
def carregar_features_com_calendario(nome_tabela: str, aplicar_corte_regime: bool = True) -> DataFrame:
    """
    Lê silver.<nome_tabela> e junta os atributos de silver.features_calendario
    pela chave data_abertura. Levanta erro se o join gerar nulo novo (sinal de
    que as duas tabelas não cobrem o mesmo período — algo mudou e precisa de
    atenção antes de treinar qualquer modelo em cima disso).

    Por padrão (aplicar_corte_regime=True), corta a série a partir de
    DATA_INICIO_TREINO_ML — ver justificativa acima. Passe False só se
    precisar do histórico completo por algum motivo específico (ex.: gráfico
    de contexto histórico, não treino de modelo).
    """
    serie = spark.table(qualified_table(SCHEMA_SILVER, nome_tabela))
    calendario = spark.table(qualified_table(SCHEMA_SILVER, "features_calendario"))

    colunas_calendario = [c for c in calendario.columns if c != "data_abertura"]
    joined = serie.join(calendario, "data_abertura", "left")

    linhas_com_nulo_novo = joined.filter(F.col(colunas_calendario[0]).isNull()).count()
    if linhas_com_nulo_novo > 0:
        raise ValueError(
            f"Join de '{nome_tabela}' com features_calendario gerou {linhas_com_nulo_novo} "
            "linhas com nulo novo — as duas tabelas não cobrem o mesmo período de datas. "
            "Rode o notebook 05 de novo antes de treinar."
        )

    if aplicar_corte_regime:
        joined = joined.filter(F.col("data_abertura") >= F.lit(DATA_INICIO_TREINO_ML))

    return joined


print("carregar_features_com_calendario() pronta.")

## `split_temporal`

Split sequencial com embargo. Defaults: 30 dias de teste, 30 de
validação, 7 de embargo em cada fronteira (cobre o horizonte D+7, o
maior alvo do projeto) — todos ajustáveis por parâmetro.

In [ ]:
def split_temporal(
    df: DataFrame,
    coluna_data: str = "data_abertura",
    dias_teste: int = 30,
    dias_validacao: int = 30,
    dias_embargo: int = 7,
) -> tuple:
    """
    Divide um DataFrame de série temporal em (treino, validacao, teste),
    sequencialmente — nunca aleatório. Aplica um embargo (dias descartados,
    não usados em nenhum dos três conjuntos) em cada fronteira, para evitar
    que o target D+7 de uma linha de treino/validação vaze para dentro do
    período coberto pelo conjunto seguinte.
    """
    data_max = df.agg(F.max(coluna_data)).first()[0]

    inicio_teste = data_max - timedelta(days=dias_teste - 1)
    fim_embargo_teste = inicio_teste - timedelta(days=dias_embargo)

    inicio_validacao = fim_embargo_teste - timedelta(days=dias_validacao - 1)
    fim_embargo_validacao = inicio_validacao - timedelta(days=dias_embargo)

    teste = df.filter(F.col(coluna_data) >= inicio_teste)
    validacao = df.filter(
        (F.col(coluna_data) >= inicio_validacao) & (F.col(coluna_data) <= fim_embargo_teste)
    )
    treino = df.filter(F.col(coluna_data) <= fim_embargo_validacao)

    return treino, validacao, teste


print("split_temporal() pronta.")

## Auto-teste — roda toda vez que este notebook é chamado

Valida as duas funções contra `features_series_prioridade` (tabela
pequena, teste rápido) antes de qualquer notebook de treino confiar
nelas. Se algo aqui falhar, os notebooks `08`-`11` não devem prosseguir.

In [ ]:
_teste_df = carregar_features_com_calendario("features_series_prioridade")

# Corte de regime: nenhuma linha antes de DATA_INICIO_TREINO_ML deveria sobrar
_data_min_pos_corte = _teste_df.agg(F.min("data_abertura")).first()[0]
assert _data_min_pos_corte.isoformat() >= DATA_INICIO_TREINO_ML, (
    f"Corte de regime falhou: menor data no resultado é {_data_min_pos_corte}, "
    f"esperado >= {DATA_INICIO_TREINO_ML}"
)

# Flags de calendário precisam sobreviver ao corte, não só ao join
assert "is_feriado" in _teste_df.columns, "is_feriado sumiu do resultado!"
assert "is_fim_de_semana" in _teste_df.columns, "is_fim_de_semana sumiu do resultado!"
_linhas_com_flag_nula = _teste_df.filter(
    F.col("is_feriado").isNull() | F.col("is_fim_de_semana").isNull()
).count()
assert _linhas_com_flag_nula == 0, f"{_linhas_com_flag_nula} linhas com flag de calendário nula após o corte!"

_treino, _val, _teste = split_temporal(_teste_df)

_datas_treino = set(r[0] for r in _treino.select("data_abertura").distinct().collect())
_datas_val = set(r[0] for r in _val.select("data_abertura").distinct().collect())
_datas_teste = set(r[0] for r in _teste.select("data_abertura").distinct().collect())

assert not (_datas_treino & _datas_val), "Overlap entre treino e validação!"
assert not (_datas_val & _datas_teste), "Overlap entre validação e teste!"
assert not (_datas_treino & _datas_teste), "Overlap entre treino e teste!"

print("Auto-teste OK:")
print(f"  Menor data após corte de regime: {_data_min_pos_corte} (>= {DATA_INICIO_TREINO_ML})")
print("  Flags is_feriado/is_fim_de_semana presentes e sem nulo em toda linha.")
print(f"  treino: {_treino.count()} linhas | validação: {_val.count()} linhas | teste: {_teste.count()} linhas")
print("  Sem overlap de datas entre os três conjuntos.")

del _teste_df, _treino, _val, _teste, _datas_treino, _datas_val, _datas_teste, _data_min_pos_corte, _linhas_com_flag_nula